# Fine-tuning Qwen2.5-VL-7B for Robot Navigation
## bf16 LoRA fine-tuning on (camera + LiDAR → intent) dataset

**Hardware:** Google Colab A100 (40GB VRAM)
**Method:** LoRA r=16 in bf16 (no quantization needed on A100)
**Framework:** HuggingFace Transformers Trainer + PEFT
**Model:** Qwen/Qwen2.5-VL-7B-Instruct
**Dataset:** Loaded directly from HuggingFace Hub
**Output:** Merged standalone model pushed to HuggingFace → served on AMD MI300X

### Pipeline
```
Colab A100 training → LoRA adapter → merge into base → HuggingFace repo
                                                              ↓
                                         AMD MI300X qwen_vl_server.py
                                                              ↓
                                              ROS2 llm_driver_node_lidar
```


In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)


## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install -q transformers==4.51.0
!pip install -q peft==0.15.1
!pip install -q accelerate==1.6.0
!pip install -q datasets==3.5.0
!pip install -q huggingface_hub
!pip install -q qwen-vl-utils
!pip install -q matplotlib
print("Dependencies installed.")


## Step 2 — Authenticate with HuggingFace

Get your token from https://huggingface.co/settings/tokens  
Use a token with **write** scope so we can also push the trained adapter back.


In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Reads HF_TOKEN from Colab Secrets (left sidebar → key icon → add HF_TOKEN)
# This avoids pasting your token in plain text in the notebook.
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in via Colab Secrets.")
except Exception:
    # Fallback: paste token manually
    login()  # will prompt interactively


## Step 3 — Configuration

In [ ]:
# ── Edit these lines ─────────────────────────────────────────────────────────
HF_DATASET_REPO = "biggestFudge/robot-navigation-dataset"
HF_ADAPTER_REPO = "biggestFudge/qwen2-5-vl-7b-robot-lora"    # intermediate
HF_MERGED_REPO  = "biggestFudge/qwen2-5-vl-7b-robot-merged-v2"  # final — point server here
# ─────────────────────────────────────────────────────────────────────────────

MODEL_ID   = "Qwen/Qwen2.5-VL-7B-Instruct"
OUTPUT_DIR = "/content/checkpoints"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Dataset : {HF_DATASET_REPO}")
print(f"Adapter : {HF_ADAPTER_REPO}")
print(f"Merged  : {HF_MERGED_REPO}  ← this is what qwen_vl_server.py will load")
print(f"Base    : {MODEL_ID}")


## Step 4 — Load Dataset from HuggingFace

In [ ]:
from datasets import load_dataset
from collections import Counter

print(f"Loading {HF_DATASET_REPO} ...")
raw = load_dataset(HF_DATASET_REPO, split="train")
print(f"Total samples: {len(raw)}")

# Class distribution
counter = Counter(raw['intent'])
print("\nClass distribution:")
for intent, count in sorted(counter.items(), key=lambda x: -x[1]):
    pct = count / len(raw) * 100
    bar = '█' * int(pct / 2)
    print(f"  {intent:<10} {count:>5}  {pct:>5.1f}%  {bar}")

print(f"\nColumns: {raw.column_names}")


## Step 5 — Convert to Qwen2.5-VL Chat Format

Each sample becomes a single-turn conversation:
- **System:** minimal navigation instruction
- **User:** camera image + LiDAR image + prompt
- **Assistant:** `{"intent": "FORWARD"}` (or LEFT/RIGHT/REVERSE/STOP)

The system prompt is intentionally minimal — the fine-tuned model encodes
driving knowledge in its weights rather than depending on a long prompt.


In [ ]:
import json

SYSTEM_PROMPT = (
    "You are a robot navigation controller. "
    "Given a forward camera image and a top-down LiDAR map, "
    "output only a JSON object with a single key 'intent' "
    "with value one of: FORWARD, LEFT, RIGHT, REVERSE, STOP."
)

USER_PROMPT = (
    "IMAGE 1: forward camera view. "
    "IMAGE 2: top-down LiDAR map (robot = cyan dot at centre facing up, "
    "rings = 4m/8m/12m, white dots = obstacles). "
    "What is the next driving action?"
)

def sample_to_conversation(sample):
    cam_url   = f"data:image/jpeg;base64,{sample['image_b64']}"
    lidar_url = f"data:image/jpeg;base64,{sample['lidar_b64']}"
    answer    = json.dumps({"intent": sample['intent']})
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": cam_url},
                    {"type": "image", "image": lidar_url},
                    {"type": "text",  "text": USER_PROMPT},
                ]
            },
            {"role": "assistant", "content": answer}
        ]
    }

print("Converting to chat format...")
conversations = [sample_to_conversation(s) for s in raw]
print(f"Converted {len(conversations)} samples.")
print("Example output:", conversations[0]['messages'][2]['content'])


In [ ]:
import random, json
from datasets import Dataset

random.seed(42)
random.shuffle(conversations)

split_idx    = int(len(conversations) * 0.95)
train_convs  = conversations[:split_idx]
val_convs    = conversations[split_idx:]

# Serialize the nested message structure to a JSON string so Arrow
# never tries to infer a schema from mixed content types.
# The collator will deserialize it back before passing to the model.
def to_hf_row(conv):
    return {"messages_json": json.dumps(conv["messages"])}

train_dataset = Dataset.from_list([to_hf_row(c) for c in train_convs])
val_dataset   = Dataset.from_list([to_hf_row(c) for c in val_convs])

print(f"Train : {len(train_dataset)} samples")
print(f"Val   : {len(val_dataset)} samples")
print(f"Columns: {train_dataset.column_names}")


In [ ]:
from collections import Counter
import json

# Check val set distribution
val_intents = []
for item in val_convs:
    msgs = item["messages"]
    answer = msgs[-1]["content"]          # {"intent": "LEFT"} etc
    import json as _json
    intent = _json.loads(answer)["intent"]
    val_intents.append(intent)

counter = Counter(val_intents)
total   = len(val_intents)
print(f"Val set size: {total}")
print()
for intent in ["FORWARD", "LEFT", "RIGHT", "REVERSE", "STOP"]:
    count = counter.get(intent, 0)
    pct   = count / total * 100
    bar   = "█" * int(pct / 2)
    print(f"  {intent:<10} {count:>4}  {pct:>5.1f}%  {bar}")

## Step 6 — Load Model in bf16 (A100)

On A100 we have 40GB VRAM — no quantization needed.
Training in bf16 gives cleaner gradients than QLoRA and produces a
higher-quality merged model for deployment on the MI300X.

- Model in bf16: ~14GB VRAM
- LoRA adapters + optimizer: ~8GB
- Activations (batch=2): ~6GB
- Total: ~28GB — comfortable on A100 40GB

Download takes ~3 minutes.


In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model

print(f"Loading {MODEL_ID} in bf16...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False  # required for gradient checkpointing

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    min_pixels=448 * 448,   # higher res than T4 — A100 can handle it
    max_pixels=448 * 448,
)

print("Model loaded.")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")


In [ ]:
# No kbit preparation needed — model is full bf16
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

# Required for gradient checkpointing + PEFT to work together:
# enables gradients on input embeddings so backprop can flow through
# the frozen base model layers to the LoRA adapters.
model.enable_input_require_grads()

model.print_trainable_parameters()


## Step 7 — Data Collator

In [ ]:
from qwen_vl_utils import process_vision_info
import json, torch

def collate_fn(batch):
    texts, image_inputs_list = [], []

    for item in batch:
        msgs = json.loads(item["messages_json"])
        text = processor.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
        image_inputs, _ = process_vision_info(msgs)
        image_inputs_list.append(image_inputs)

    # Single processor call for the whole batch — avoids token count mismatch
    # that occurs when calling the processor twice with different inputs.
    inputs = processor(
        text=texts,
        images=image_inputs_list,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    )

    # Find the assistant response boundary using the im_end token id.
    # Everything after the LAST <|im_end|> of the user turn is the assistant
    # response — we train only on those tokens.
    labels = inputs["input_ids"].clone()
    labels[:] = -100   # mask everything by default

    # im_start token id — used to find assistant turn boundary
    im_start_id = processor.tokenizer.convert_tokens_to_ids("<|im_start|>")
    assistant_id = processor.tokenizer.convert_tokens_to_ids("assistant")

    for i in range(labels.shape[0]):
        ids = inputs["input_ids"][i].tolist()
        # Find the last occurrence of <|im_start|> followed by "assistant"
        # That marks the start of the assistant turn we want to train on.
        for pos in range(len(ids) - 1, 0, -1):
            if ids[pos] == im_start_id and pos + 1 < len(ids) and ids[pos + 1] == assistant_id:
                # Unmask from after "assistant\n" (skip im_start + "assistant" + newline = +3)
                labels[i, pos + 3:] = inputs["input_ids"][i, pos + 3:]
                break

    inputs["labels"] = labels
    return dict(inputs)

print("Collator ready.")


## Step 8 — Train

A100 config vs T4:
- No quantization → cleaner gradients
- batch_size=2 (vs 1 on T4)
- bf16 instead of fp16
- 448×448 images (vs 224×224)
- Expected time: ~45-60 minutes for 3 epochs


In [ ]:
from transformers import TrainingArguments, Trainer
import torch

class VLMTrainer(Trainer):
    """
    Thin Trainer subclass that removes the `num_items_in_batch` kwarg
    injected by newer Transformers versions before it reaches the model.
    Qwen2_5_VLForConditionalGeneration.forward() does not accept it.
    """
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,
    optim="adamw_torch",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    fp16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataloader_num_workers=2,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = VLMTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)

steps_per_epoch = len(train_dataset) // (training_args.per_device_train_batch_size
                                         * training_args.gradient_accumulation_steps)
print(f"Steps per epoch : {steps_per_epoch}")
print(f"Total steps     : {steps_per_epoch * training_args.num_train_epochs}")
print("\nTraining...")
trainer.train()


## Step 9 — Save Adapter and Push to HuggingFace

In [ ]:
import matplotlib.pyplot as plt, os

# Save adapter locally
adapter_local = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(adapter_local)
processor.save_pretrained(adapter_local)
print(f"Adapter saved locally: {adapter_local}")

# Push adapter to HuggingFace (backup — merged model is the real deliverable)
print(f"\nPushing adapter to {HF_ADAPTER_REPO} ...")
model.push_to_hub(HF_ADAPTER_REPO, private=True)
processor.push_to_hub(HF_ADAPTER_REPO, private=True)
print("Adapter pushed.")

# Loss curve
log_history  = trainer.state.log_history
train_losses = [(x['step'], x['loss'])      for x in log_history if 'loss'      in x]
eval_losses  = [(x['step'], x['eval_loss']) for x in log_history if 'eval_loss' in x]

if train_losses:
    fig, ax = plt.subplots(figsize=(10, 4))
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label='train loss', alpha=0.8)
    if eval_losses:
        e_steps, e_losses = zip(*eval_losses)
        ax.plot(e_steps, e_losses, label='eval loss', marker='o', linewidth=2)
    ax.set(xlabel='Step', ylabel='Loss', title='Training Loss — Qwen2.5-VL-7B Robot LoRA')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'loss_curve.png'), dpi=150)
    plt.show()
    print(f"Final train loss : {losses[-1]:.4f}")
    if eval_losses:
        print(f"Final eval loss  : {e_losses[-1]:.4f}")


## Step 10 — Merge Adapter into Base Model and Push to HuggingFace

This creates the **final deployable model** that `qwen_vl_server.py` on the MI300X
will load directly — no PEFT, no adapter, just a standard Qwen2.5-VL model
that happens to know how to drive your robot.

On A100 with 40GB this merge runs entirely on GPU and takes ~3 minutes.

After this cell completes, update your MI300X `.env`:
```
LLM_BACKEND=local
LOCAL_BASE_URL=http://your-mi300x-ip:8001/v1
LOCAL_MODEL=YOUR_HF_USERNAME/qwen2-5-vl-7b-robot-merged-v2
```


In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

# Define your paths
base_model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
adapter_id = "biggestFudge/qwen2-5-vl-7b-robot-lora"  # Your adapter repo
merged_repo_id = "biggestFudge/qwen2-5-vl-7b-robot-merged-v2" # Your final target repo

print("1/4: Loading base model...")
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print("2/4: Loading adapter from HF and merging...")
# This pulls your adapter from your HF repo and attaches it to the base model
model = PeftModel.from_pretrained(base_model, adapter_id)

# Perform the merge: this incorporates the LoRA weights into the base weights
merged_model = model.merge_and_unload()

print("3/4: Loading processor...")
# We need to grab the processor from the base model so we can push it to the new repo
processor = AutoProcessor.from_pretrained(base_model_id, trust_remote_code=True)

print(f"4/4: Pushing merged model to {merged_repo_id}...")
# Push the weights
merged_model.push_to_hub(
    merged_repo_id, 
    safe_serialization=True,
    private=False 
)
# Push the processor (essential for the vision part of the model to work)
processor.push_to_hub(merged_repo_id, private=False)

print("\nSUCCESS: Your model is now merged and hosted as a full standalone model.")
print(f"Update your server to use: {merged_repo_id}")

## Step 11 — Quick Inference Test

In [ ]:
# ════════════════════════════════════════════════════════════════
# DIAGNOSTIC — Test fine-tuned model on real dataset samples
# ════════════════════════════════════════════════════════════════
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from datasets import load_dataset
from collections import Counter
import torch, json, base64, random
from PIL import Image
from io import BytesIO

MERGED_MODEL_ID = "biggestFudge/qwen2-5-vl-7b-robot-merged-v2"
HF_DATASET_REPO = "biggestFudge/robot-navigation-dataset"
N_SAMPLES       = 30

# ── Step 1: Load model ────────────────────────────────────────────────────────
print("Loading model...")
diag_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MERGED_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
diag_model.eval()

# ── CRITICAL CHECK: verify this is a proper merged model, not a PeftModel ────
from peft import PeftModel
if isinstance(diag_model, PeftModel):
    print("\n!!! WARNING: Model loaded as PeftModel — merge was NOT successful !!!")
    print("The HuggingFace repo contains adapter files, not a merged model.")
    print("This explains the FORWARD bias — the LoRA adapters are not applied.")
else:
    print("✓ Model is a standard merged model (not PeftModel)")

# Also check if adapter files exist in the repo
from huggingface_hub import list_repo_files
repo_files = list(list_repo_files(MERGED_MODEL_ID))
adapter_files = [f for f in repo_files if "adapter" in f.lower() or "lora" in f.lower()]
if adapter_files:
    print(f"\n!!! Adapter files found in repo: {adapter_files}")
    print("This confirms the model was NOT properly merged before pushing.")
else:
    print(f"✓ No adapter files in repo — clean merged model")

diag_processor = AutoProcessor.from_pretrained(
    MERGED_MODEL_ID,
    trust_remote_code=True,
    min_pixels=448 * 448,
    max_pixels=448 * 448,
)
print("Processor loaded.")

# ── Step 2: Load dataset samples ─────────────────────────────────────────────
print(f"\nLoading {N_SAMPLES} samples from {HF_DATASET_REPO}...")
raw = load_dataset(HF_DATASET_REPO, split="train")
random.seed(42)
indices    = random.sample(range(len(raw)), N_SAMPLES)
samples    = [raw[i] for i in indices]
gt_counter = Counter(s["intent"] for s in samples)
print("Ground truth distribution:", dict(gt_counter))

# ── Step 3: Run inference ─────────────────────────────────────────────────────
print(f"\nRunning inference on {N_SAMPLES} samples...")
SYSTEM_PROMPT = (
    "You are a robot navigation controller. "
    "Given a forward camera image and a top-down LiDAR map, "
    "output only a JSON object with a single key 'intent' "
    "with value one of: FORWARD, LEFT, RIGHT, REVERSE, STOP."
)
USER_PROMPT = (
    "IMAGE 1: forward camera view. "
    "IMAGE 2: top-down LiDAR map (robot = cyan dot at centre facing up, "
    "rings = 4m/8m/12m, white dots = obstacles). "
    "What is the next driving action?"
)

correct      = 0
pred_counter = Counter()
mismatches   = []

for s in samples:
    # Decode images from base64 to PIL — pass as PIL directly to processor
    cam_img   = Image.open(BytesIO(base64.b64decode(s["image_b64"]))).convert("RGB")
    lidar_img = Image.open(BytesIO(base64.b64decode(s["lidar_b64"]))).convert("RGB")

    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": cam_img},    # PIL Image directly
            {"type": "image", "image": lidar_img},
            {"type": "text",  "text": USER_PROMPT},
        ]},
    ]

    text = diag_processor.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = diag_processor(
        text=[text],
        images=[cam_img, lidar_img],
        return_tensors="pt"
    ).to(diag_model.device)

    with torch.no_grad():
        out_ids = diag_model.generate(
            **inputs, max_new_tokens=32, temperature=0.1, do_sample=False
        )

    response = diag_processor.decode(
        out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    try:
        pred = json.loads(response).get("intent", "?")
    except Exception:
        pred = response[:20]

    pred_counter[pred] += 1
    gt = s["intent"]
    if pred == gt:
        correct += 1
    else:
        mismatches.append((gt, pred))

# ── Step 4: Results ───────────────────────────────────────────────────────────
print(f"\nAccuracy: {correct}/{N_SAMPLES} = {correct/N_SAMPLES*100:.1f}%")

print("\nPredicted distribution:")
for intent, count in sorted(pred_counter.items(), key=lambda x: -x[1]):
    pct = count / N_SAMPLES * 100
    bar = "█" * int(pct / 3)
    print(f"  {intent:<10} {count:>3}  {pct:>5.1f}%  {bar}")

print("\nGround truth distribution:")
for intent, count in sorted(gt_counter.items(), key=lambda x: -x[1]):
    pct = count / N_SAMPLES * 100
    bar = "█" * int(pct / 3)
    print(f"  {intent:<10} {count:>3}  {pct:>5.1f}%  {bar}")

if mismatches:
    print(f"\nMismatches ({len(mismatches)}):")
    for gt, pred in mismatches[:10]:
        print(f"  GT: {gt:<10} → Pred: {pred}")
